# explainer — Regression Example

This notebook demonstrates every feature of the `explainer` package using a **regression** task.

**Dataset:** California Housing (sklearn built-in)  
**Primary model:** LightGBM `LGBMRegressor`  
**Also shown:** `Ridge` (LinearExplainer path)  

**Features covered:**
- Basic SHAP explanations (aggregate + per-observation)
- All 10 visualisation types
- Interaction effects
- Bootstrap confidence intervals
- Feature groups
- Linear model explainability (Ridge)
- Saving plots to disk

In [ ]:
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

from transpyrancy import Explainer

warnings.filterwarnings('ignore')
print('explainer ready ✓')

---
## 1 · Load Data & Train LightGBM

In [ ]:
housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=63,
    random_state=42,
    verbose=-1,
)
model.fit(X_train, y_train)

preds = model.predict(X_test)
print(f'Test MAE : {mean_absolute_error(y_test, preds):.4f}')
print(f'Test R²  : {r2_score(y_test, preds):.4f}')
print(f'Features : {X.shape[1]}')
print(X.columns.tolist())

---
## 2 · Construct the Explainer

In [ ]:
feature_groups = {
    'location':    ['Latitude', 'Longitude'],
    'income':      ['MedInc'],
    'house_size':  ['AveRooms', 'AveBedrms'],
    'occupancy':   ['Population', 'AveOccup'],
    'age':         ['HouseAge'],
}

exp = Explainer(
    model_type='lightgbm',
    model=model,
    background_data=X_train,
    task='regression',
    feature_groups=feature_groups,
)
print(exp)

---
## 3 · Explain the Test Set

In [ ]:
# Use a subsample for speed in the notebook
X_sample = X_test.iloc[:200]
y_sample = y_test.iloc[:200]

result = exp.explain(
    X_sample,
    compute_interactions=True,
    compute_confidence_intervals=True,
    n_bootstrap=50,
    ci_level=0.95,
)

print(result)

---
## 4 · Inspect the Data

In [ ]:
print('SHAP values shape:', result.shap_values.shape)
result.shap_values.head(3)

In [ ]:
print('Global feature importance (mean |SHAP|):')
result.aggregate

In [ ]:
print('Top 5 features:')
result.top_features(5)

In [ ]:
print('Predictions vs actuals (first 10):')
comparison = pd.DataFrame({
    'actual': y_sample.values[:10],
    'predicted': result.predictions[:10],
    'error': result.predictions[:10] - y_sample.values[:10],
})
comparison

In [ ]:
print(f'Expected value (baseline): {result.expected_value:.4f}')
print(f'Mean actual target       : {y_sample.mean():.4f}')

---
## 4.5 · Built-in Help System

`result.help()` provides plain-English explanations of every attribute and plot — context-aware and Jupyter-friendly.

In [ ]:
# Full overview — what's in this result, what each thing means
result.help()

In [ ]:
# Zoom in on a specific attribute
result.help('shap_values')

In [ ]:
# Get help on a specific plot
result.help('plot_dependence')

In [ ]:
# Short aliases work too
result.help('ci')       # confidence intervals
result.help('groups')   # feature group contributions

In [ ]:
# Pre-explain overview of the library itself
Explainer.help()

---
## 5 · Visualisations

### 5a · Summary (Beeswarm)

In [ ]:
result.plot_summary(max_display=8)

### 5b · Feature Importance Bar Chart

In [ ]:
result.plot_feature_importance(top_n=8)

### 5c · Waterfall — Best and Worst Predictions

In [ ]:
errors = np.abs(result.predictions - y_sample.values)

best_idx  = int(np.argmin(errors))
worst_idx = int(np.argmax(errors))

print(f'Best prediction  — index {best_idx}: actual={y_sample.values[best_idx]:.3f}, pred={result.predictions[best_idx]:.3f}')
result.plot_waterfall(index=best_idx, max_display=8)

In [ ]:
print(f'Worst prediction — index {worst_idx}: actual={y_sample.values[worst_idx]:.3f}, pred={result.predictions[worst_idx]:.3f}')
result.plot_waterfall(index=worst_idx, max_display=8)

### 5d · Force Plots

In [ ]:
result.plot_force(index=best_idx)

In [ ]:
result.plot_force(index=worst_idx)

### 5e · Decision Plot

In [ ]:
result.plot_decision(max_display=8)

### 5f · SHAP Heatmap

In [ ]:
result.plot_heatmap(max_display=8)

### 5g · Dependence Plots

In [ ]:
# How does median income drive the prediction?
result.plot_dependence('MedInc')

In [ ]:
# Latitude vs Longitude interaction
result.plot_dependence('Latitude', interaction_feature='Longitude')

### 5h · Feature Group Contributions

In [ ]:
print('Group contributions (first 5 rows):')
print(result.group_contributions.head())
result.plot_group_contributions()

### 5i · Confidence Intervals

In [ ]:
# Aggregate CIs
result.plot_confidence_intervals(top_n=8)

In [ ]:
# CI for the worst-predicted observation
result.plot_confidence_intervals(observation_index=worst_idx, top_n=8)

### 5j · Interaction Effects

In [ ]:
print('Interaction effects shape:', result.interaction_effects.shape)
result.plot_interaction_heatmap('MedInc', top_n=7)

In [ ]:
result.plot_interaction_heatmap('Latitude', top_n=7)

---
## 6 · Diagnosing Systematic Errors

Combine SHAP values with prediction errors to find where the model struggles.

In [ ]:
import matplotlib.pyplot as plt

errors = result.predictions - y_sample.values
shap_df = result.shap_values

# Correlation between SHAP values and signed prediction error
corr = shap_df.corrwith(pd.Series(errors, index=shap_df.index)).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in corr.values]
ax.barh(corr.index[::-1], corr.values[::-1], color=colors[::-1])
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Correlation with prediction error')
ax.set_title('Which SHAP values correlate with model mistakes?', fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

---
## 7 · Save All Plots to Disk

In [ ]:
result_saved = exp.explain(
    X_sample,
    visualize=False,
    save_plots='./plots/regression/',
)

import os
files = sorted(os.listdir('./plots/regression/'))
print(f'{len(files)} PNGs saved:')
for f in files:
    print(' ', f)

---
## 8 · Linear Model — Ridge Regression

The `LinearExplainer` path works identically — just requires `background_data`.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Convert back to DataFrame to keep feature names
X_train_sc = pd.DataFrame(X_train_sc, columns=X.columns)
X_test_sc  = pd.DataFrame(X_test_sc,  columns=X.columns)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_sc, y_train)

ridge_preds = ridge.predict(X_test_sc)
print(f'Ridge R² : {r2_score(y_test, ridge_preds):.4f}')

In [ ]:
ridge_exp = Explainer(
    model_type='ridge',
    model=ridge,
    background_data=X_train_sc,
    task='regression',
)
print(ridge_exp)

ridge_result = ridge_exp.explain(X_test_sc.iloc[:100])
print(ridge_result)

In [ ]:
ridge_result.plot_summary()

In [ ]:
ridge_result.plot_waterfall(index=0)

In [ ]:
# For a linear model, the SHAP values are simply weight × (feature_value - mean)
# So the sum of SHAP values + expected_value ≈ prediction
obs = 0
shap_sum = ridge_result.shap_values.iloc[obs].sum()
pred = ridge_result.predictions[obs]
ev   = ridge_result.expected_value
print(f'Expected value + SHAP sum : {ev + shap_sum:.4f}')
print(f'Model prediction          : {pred:.4f}')
print(f'Difference (should ≈ 0)   : {abs(ev + shap_sum - pred):.6f}')

---
## 9 · Explaining a Single Row

In [ ]:
# The most expensive house in the test set
richest_idx = int(y_test.values.argmax())
expensive_house = X_test.values[richest_idx]  # 1-D numpy array

single = exp.explain(expensive_house)

print(f'Actual price (×$100k): {y_test.values[richest_idx]:.3f}')
print(f'Predicted price      : {single.predictions[0]:.3f}')

single.plot_waterfall(index=0)